<a href="https://colab.research.google.com/github/adarshpatel39/MACHINE-LEARNING-PROJECTS/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
import string

# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  # 1. Preprocess the input text
  def preprocess_text(text):
      text = text.lower()
      text = text.translate(str.maketrans('', '', string.punctuation))
      return text

  preprocessed_text = preprocess_text(pred_text)

  # 2. Tokenize the preprocessed text
  sequence = tokenizer.texts_to_sequences([preprocessed_text])

  # 3. Pad the tokenized sequence
  padded_sequence = tf.keras.preprocessing.sequence.pad_sequences(
      sequence, maxlen=maxlen, padding='post', truncating='post'
  )

  # 4. Make a prediction using the trained model
  prediction_probability = model.predict(padded_sequence)[0][0]

  # 5. Convert the prediction probability into a 'ham' or 'spam' label
  if prediction_probability >= 0.5:
    label = 'spam'
  else:
    label = 'ham'

  return [float(prediction_probability), label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

# Task
Load and prepare the training and validation data for SMS spam classification from "train-data.tsv" and "valid-data.tsv" into pandas DataFrames, and inspect their structure.

## Load and Prepare Data

### Subtask:
Load the training and validation data from the .tsv files into pandas DataFrames. Inspect the data structure and content to understand its format and identify columns for text and labels.


**Reasoning**:
Load the training and validation data from the .tsv files into pandas DataFrames as instructed, then inspect their structure and content.



In [ ]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'text'])
valid_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'text'])

print("Train DataFrame Head:")
print(train_df.head())
print("\nValid DataFrame Head:")
print(valid_df.head())

print("\nTrain DataFrame Info:")
train_df.info()
print("\nValid DataFrame Info:")
valid_df.info()

## Preprocess Text Data

### Subtask:
Clean and preprocess the text messages. This will involve converting text to lowercase, removing punctuation, and tokenizing the text. Then, convert the text into numerical sequences suitable for a neural network using `tf.keras.preprocessing.text.Tokenizer` and `tf.keras.preprocessing.sequence.pad_sequences`.


**Reasoning**:
The first instruction is to convert the 'label' column from categorical strings ('ham', 'spam') to numerical format (0 for 'ham', 1 for 'spam') for both the training and validation dataframes. The second instruction is to define a function to preprocess text messages by converting them to lowercase and removing punctuation, then apply this function to the 'text' column of both dataframes. I will perform these two steps in a single code block.



In [ ]:
import string

# 1. Convert labels to numerical format
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
valid_df['label'] = valid_df['label'].map({'ham': 0, 'spam': 1})

# 2. Define a function to preprocess text messages
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# Apply the preprocessing function to the 'text' column
train_df['text'] = train_df['text'].apply(preprocess_text)
valid_df['text'] = valid_df['text'].apply(preprocess_text)

print("Train DataFrame after label conversion and text preprocessing:")
print(train_df.head())
print("\nValid DataFrame after label conversion and text preprocessing:")
print(valid_df.head())

**Reasoning**:
The next step is to initialize a `tf.keras.preprocessing.text.Tokenizer`, fit it on the training data, convert text to numerical sequences for both training and validation sets, determine an appropriate `maxlen` for padding based on the training data, and then pad both training and validation sequences to this `maxlen`.



In [ ]:
max_words = 10000 # Max number of words to keep, based on word frequency.
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(train_df['text'])

train_sequences = tokenizer.texts_to_sequences(train_df['text'])
valid_sequences = tokenizer.texts_to_sequences(valid_df['text'])

# Determine maxlen for padding based on training sequences
sequence_lengths = [len(x) for x in train_sequences]
maxlen = int(np.percentile(sequence_lengths, 95)) # Use 95th percentile as maxlen

print(f"Maximum sequence length for padding: {maxlen}")

train_padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    train_sequences, maxlen=maxlen, padding='post', truncating='post'
)
valid_padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    valid_sequences, maxlen=maxlen, padding='post', truncating='post'
)

train_labels = np.array(train_df['label'])
valid_labels = np.array(valid_df['label'])

print(f"Shape of training padded sequences: {train_padded_sequences.shape}")
print(f"Shape of validation padded sequences: {valid_padded_sequences.shape}")
print(f"Shape of training labels: {train_labels.shape}")
print(f"Shape of validation labels: {valid_labels.shape}")


## Build and Compile Model

### Subtask:
Design and build a deep learning model using TensorFlow/Keras. A common approach for text classification is to use an Embedding layer followed by recurrent layers (e.g., LSTM or GRU) or convolutional layers (e.g., Conv1D), and then dense layers for classification. Compile the model with an appropriate loss function and optimizer.


**Reasoning**:
I will define and compile the deep learning model according to the instructions, using the pre-defined `max_words` and `maxlen` values, an `Embedding` layer, a `GlobalAveragePooling1D` layer, a `Dense` hidden layer, and a `Dense` output layer, then compile it with 'adam' optimizer and 'binary_crossentropy' loss.



In [ ]:
embedding_dim = 16 # You can experiment with different dimensions

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(max_words, embedding_dim, input_shape=(maxlen,)),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'), # Hidden layer with 24 units
    tf.keras.layers.Dense(1, activation='sigmoid') # Output layer for binary classification
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Reasoning**:
The previous code successfully built and compiled the model, and the `model.summary()` output showed correct parameter counts, indicating the model was correctly instantiated despite the Keras `UserWarning`. Therefore, the model building step is complete, and the next logical step is to train the model.



In [ ]:
epochs = 10 # You can experiment with different numbers of epochs
batch_size = 32 # You can experiment with different batch sizes

history = model.fit(
    train_padded_sequences,
    train_labels,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(valid_padded_sequences, valid_labels)
)

print("Model training complete.")

# Task
Implement the `predict_message` function. This function should take a text message, preprocess it using the same steps as the training data (lowercase, remove punctuation), tokenize it using the fitted tokenizer, pad the sequence to `maxlen`, make a prediction using the trained model, and convert the numerical prediction into a 'ham' or 'spam' label.

## Implement predict_message function

### Subtask:
Implement the `predict_message` function. This function should take a text message, preprocess it using the same steps as the training data (lowercase, remove punctuation), tokenize it using the fitted tokenizer, pad the sequence to `maxlen`, make a prediction using the trained model, and convert the numerical prediction into a 'ham' or 'spam' label.


## Test predict_message function

### Subtask:
Execute the provided test function `test_predictions()` to verify the correctness of the `predict_message` function and the model's performance on new messages.


**Reasoning**:
The subtask requires executing the `test_predictions()` function to verify the model's performance. This function is already defined in the notebook, so I will add a new code cell to explicitly call it.



In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()